# SGBR E1 trên Kaggle: chạy theo shard (6 người)

Toàn bộ thí nghiệm E1 trong phạm vi đã duyệt (13 bộ dữ liệu, 13 phương pháp, seed 1–5, Axgb chỉ trên Core-10) gồm **686 run**, khoảng **145 giờ CPU** ở tốc độ máy của tác giả. Các run được chia cho **6 shard** sao cho tải gần bằng nhau (ước tính mỗi shard ≈ 5,9–6,4 giờ ở tốc độ tác giả; Kaggle nhanh hơn một chút, σ ≈ 0,84).

**Mỗi người chỉ làm 4 bước:**
1. Kaggle → **New Notebook → File → Import Notebook**, chọn file này.
2. Settings: **Accelerator → None**, **Internet → On**. Không cần GitHub token.
3. Ở ô cấu hình bên dưới, đổi `SHARD` thành số của mình (1 đến 6).
4. Chọn **Save Version → Save & Run All (Commit)**. Chế độ này chạy nền, được tắt trình duyệt. Khi xong, vào tab **Output** của version và tải `E1_shard<N>.zip` gửi lại.

| Shard | Nội dung chính |
|---|---|
| 1, 2 | 4 run NZEnergy dài nhất mỗi shard (Sgb(Srp), Srp(Sgbt)) |
| 3–6 | Các run NZEnergy còn lại và toàn bộ 12 bộ dữ liệu khác |

**Lưu ý:**
- Giới hạn của Kaggle là 12 giờ mỗi session. Notebook chạy run dài trước và tự dừng nhận run mới khi hết `SESSION_BUDGET_HOURS`.
- Nếu shard chưa xong, chạy lại version mới với cùng `SHARD` và gắn file zip cũ làm input (`RESUME_ZIP`); các run đã xong sẽ được bỏ qua.
- Lệnh MOA, dữ liệu và seed lấy nguyên từ repo tác giả (commit `f62f7b4`). Chỉ thay `-Xmx48g` → `-Xmx6g` (Kaggle có 30 GiB cho 4 JVM chạy song song); heap được ghi lại cho từng run.
- Shard nào gặp `oom` thì báo lại để chạy riêng run đó với heap lớn hơn.


In [ ]:
SHARD = 1                # <-- each person: 1, 2, 3, 4, 5 or 6
NUM_SHARDS = 6
JOBS = 4                 # Kaggle CPU sessions have 4 vCPUs
XMX = "default=6g"       # 4 JVMs x 6 GiB fit the 30 GiB cgroup limit
SESSION_BUDGET_HOURS = 11.0
RESUME_ZIP = None        # Optional: path to an earlier E1_shard<N>.zip attached as input


In [ ]:
import base64, hashlib, sys, zlib
from pathlib import Path

WORK = Path("/kaggle/working")
SCRIPTS = WORK / "sgbr_scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)
EMBEDDED = {
 "kaggle_environment_smoke.py": "eNqdGu1y2zbyf2byDijbachYoiwnaXK6Uzu6xEncxIkndnK9sVUOREISY5JgAVK2k/rdbxcASZCi3PTUSU0Cuwtgv3fB778blVKMFnE2YtmG5DfFmmeP7t9zHOeYUVkKRih5G2flNcl4wRacXxLJpIx5RmgWkQ0T8fKGFGtakLiQ5NcXb0hIMyLKjIQ8Yj4Qun/v/r04zbkoCBWrnArJ6oGIFqyI02bgs+RZ/ZLSYl2/cFk/5gktllyk9YBoCMh1WcRJ81oucsFD2HI9VLA0X8ZJg1KKJIkXPhOCi+6gYH+UTBb37y0FT0kOG4JRYmBO1P7wv48np2cfDmfHZEqcdVHkcjIareJiXS78kKejrLyi2arMqGSXVNDR6at/f3AarOD5++PjozNEXv50sHy6ePzo6ZPxAds/2F8+eRo+XoTR/qNnzx6Hj58ePGYRfbIYA/bx7NPhu2D24ezo5ew5Irv37xH41RsQLOd+Sjcs82lOwzXzuViN1MDBCB+B17S4ydmIS4n/hiAclhWjp9bL8Kmf8xSW8+7fO/nvyVFwfHg2ezE7m9lHzW/yWBHHh1FI85uU0xHKEhDfzF69ensY/Of9hzdH714BGrLNdUaXdLVK2OiKi8s4Wzme5mTElqg9LrAtBQ0bkPAqmr7jGfMm+niFuDFP+AOwPGEFi4BuI2sfKTRABrChB//gYc3Cy+lLmkgGLzQvQNsDXhZ5WUzPRAmDBbuuH0FJYW76aL8h6+lHdh2yvCDu+9ND1KCBvY8zjXZ4nceCRR6hkig1s04gGCycka8Ou46LAI3GmRA88IA4ejfwLgvhKkTvVmNWWA2dFn7NFl8D4vDAgq0Juw2kLCIYJnukNQSrevBXxLnrGQq3jaSUAgRw2gVzwWL6ZXQFhtCxJx9eec4yRGqYO95XHBJM5jyTbNKWYM0oXDSQBS1KCSeooH09ctsSim3a/uuzsxMlojvF0KauMZF9tzuk/YkmJVPP3t+gawSsoLvyrZgbrgQv8yCJU3CtbsrSBRNyHedTjZzyMivibMlb5gEO9wOj4JljGS8SRjbj0eYAfDWoYwEn18SI8mbFmhGjqA8koWFR0kRTldpvI714SZqVSSzVzq3zWZO1ZSPRkWTJcqSP4HggdxoFaE+u19CtDtBHtp7roVpP9hE2GwK8r0ZkS3XujJE4s7bryzyJCxyXrmetHIAz4FkheJIA4MBgADkE1UiuM3EG5MBrkHCJBgsXsmhUSAPH66i02c15AzyvzqunlFNESCO3KTmf64EwL4M/Sl5QexCQuLgJwJnZoxCkMWDjkGQ1n1pcqRi6iykKYEAwbMobCRG0ww8yJMCRscUR8OngY4GZPC/M4g22QfPOJ4/mDQqoBGJhpoGbch2jPWAn+ulgi4HIOMhOWGengeC8MBYSYNBuomP1U0wWcIBy4Qrn4sI93x8+nX99dOvBeglNFxHF/CNcgzNdCzfOCle9no/nA/LM85AZLIk8r00WuaomarbWZ300eTLfCh5KPgewPXXy6bQ5qh3jalVCrXacW2QVYDEIXUqkhseWonnkR4AEJUHuabVwbu/SVwkJDYtce60fK/30L9kN6EO/9sKWetS4DVk7ETyhMectWeJPnR0p1pJrw7CkJuXHMhAMMsF4w4KCu43gv40wGVWEdlDprrwVkPoVULEXNR1ltY3gYNBFwUBmdt0IR715gx54lGGFEi6lNvoAgsiA1GM5pOE8wsFeEkb+SMWsptxJEGfB4qZgQKqLdrtblqg6EWQzIYSTG2WnircD8lD99XXaKPuEALJD266x+0WIwunDxt8C3P3l9tQGY7Dt8rdNMmEZTZWzU8LZQb+duPQvc14RQ1/tNqwY1au0olKVPPWTNemEySbuWLtf0QxX9cZ2YFfu36dR5B7YrmO8Y1M62EDRABla5H51UCImSamPCw7w4UO97m0PGaWlA6IVE/ikIf0VOCtb+YdjMoYgblzkDjI96B1DUI896PXy6L7Ni3X+PqqNKcH29h3P62W43td34K7xIKoUxiX0PsjPZF8NmRXhdYdomiBeMduiMjL4PTvQVtzhi+VIBn0TXZvXe+874Pcm9YTkEfLXHDJstGhSZooE1FpmfZXYU6iKV2xIk3iVwdTb9+9eBcez3/xevhnELcbpcY/8ixw8fPjTLnZZ6Y3NL4Nc50o9tZGxAVBiZ32zEHHk4G7AWt3KOlBqY60WGdotnFM0kwOVn9pe0tFGAhT1gz1VixXKMWAewKQxVLS1sAcE0ntaJsV0i6o5opaVFpTGts7ej24XDrASZnN1CteTSAPMzgwaqp0EvSmkR0tIS2iSQIr0uxom6N8u5MMJ/HP9Pe8H0COkM0Dg449vz47eHr07rFJMVe1lYcy2qcH65Pj1l28lVZdRenPojXTCgumPHsOczOELycSGRUG6/tIBorm7TDjFNLbZl+fZFdfnTQoBSfJShFYCvEzoSh2gzNxz5zPdUOW6fvttsnciQAFf4vzLOKOJGq90bW62DiULuEzVoYjisHAtPlhlovM7sOHiau9C7rmzECPiiS7NuHiOYfHPY3r9mtH8NP7C/vwoGYC+eu4h+36ZIgMvTvcgc7UyPNzUeVXoz9tMNXCet9NgFDoy0Gze1lHgJoQiNd1liVzzq1MzPdG5focnNiGjxn2EKt4+122bt1BrKDb3U7NEmFIwFipWm1ZVrLqOmKpWHUh/JlZlCm7tRM24EZMhhGlMo6dBEPEwCDwbFYNnQA2O6wyHhrEDgg20KVpWY5bazkCR8HAj2H+8hOobO6yx4BlS8FVvzLt7iYSHSqWgMsdtOVjAQ54kSgaDa5bkU+cTuF3soRLYjowjRt6opppzN2HTwR0qLzNcg7rL6iDKQv4OtjaWekN6ffLxiGCayEOFRXH/Kg8r1lDqGwJ/sUnUhaFWP1wN28/1Kh8YlYaigchWEIiw+5yxK4br0Iw8+4sVwAL51TCNYTfZaqj6ondz+yW2jiANgngXkeP3M/KZim9ao8wuM36VtVnXv5YhBzTQ5Riq6g/SlUq1mxYKptM47CtdmbRqaZxr914x34YEzu1m12YV1YACvS0zTJO1JAcox5xKSYw+KqZTIJ4Njaw3WgctieLPHFlHMm14ELSRG0E1ZbhhQDAVUECt4UBpZ9Ug2jpfexUk0Snzv50i/uBoSBXvHYBX4KUhC3B3kfC2cHcu9q8p2e9dqQdDG5TVKLhbWAxSjEi18ow4quuZhiXYODJmWudIpusOgum057eUSudE6NL88CqqcoSCc5UifMUH477xcaACz9zT1l4PqDqxOYfr6Jsm5c1rdw7x221CwNZo2Bm2yaWbbBtppWysu0CZfYlzDVwTqbrqBlAnK5JREa5d8aAaddxfJuML3/vFvYj2vAcDzQQTteZNrDUswuEgpZ+5MAWIoXM+BvZgFWLIKv42uqiS6uqOyj9jeNVExc2LqvByIR9fxtfgNFYLMYSYMoTDgFVPjUhVF7ou0yzlUVGpngAGnGDb3le79/wrAbqu08C2vj0IEzR/BUy+krxcJHGITrCAPxseRzrknkKVm63O50pzPAB80CFzqjt/wCQ/x8ieZK6D92DBr7NPs+D9G8f757fgfMBOZcqwsjGPUGLDEeKELpImZ8J21f9PD2qTY5VvKyq35PYiA3EzvEaBQ06dslgOnzm9zTy8PAHBBep2qZU0Kv21eD7Xd1FNRd0QSWiZheuaxpbray1ybt38zLHB1i2fOtQ6qVaojMGvN3fXvlT77Y7FjVNVXsdSfxhGn9I39fOUjMed/X5PTpW3GqIFqGqE0ERywq6ZCGPwb8rdLcosSiAOm+0IAqakWi05DGImEKcQp6X/N3hxt3BMrqyVpOUiWmm8rdOY0sMC4+E/5uf78L+H3vbID3bWbm+vcSiqVrU3rlyG4/Qn9Zb34XjGFmZXFO0Fu4qkQKoTW6hmiQ1eA3Rqpjo1MRuwZAuQfpljuurWBhvktcVO0UeaxdBHDsCxXAem7lWlcAviYO61CiyMoOq8VugCBbO98JQ8wyFLXt+opICHQwvw9zodUJPNupicei32Q+zR16K1kplolMihgCOpZBZCE3J8qDg+INXnADDx+nD2og4kdbfHrt9hcFf9vuayqPh2GS9M9LEimvM7eLczXkDGiOUmRjNnYJbpVN8oCE01iiVyV39f4eNbUEqwMLeKO2azptppd79NngJ1OS2Csgih4qs+/fDrB8jk3PoF//cFZOEDtAfJGMcPPmjhtgpIq6LC1o4umFD5uumLoz82sE3Nqb4iAczq0a8e2svoZCVomkgNfGumjZXwVQwbCMJc3fhy6WNfJsRivg1IlyrTvKkgsS0F0BKMMgogLlXz7r7XwuvIGTA7I62elL5Lm3Rulb1O3woB6u6RPdcyc4CCd3vaKEGAegGzrZ5xpSCg1wUqXd3XQmBfDcHUUjDWnsGR23bjYSu1BthdWfduTJMMT3an3zauyvIAWP1tsaT2EMiP+mULpO0nqmV7prYwTWwzH3TUQXcLTntxVB3LndtQGStQCiiZdjx0yhxExGgaoMdagysBmNp5dW6FHFUjByGYmwBDm7Q+/2h/jtS9UHLwy6AORutzIhuhJXR2nasOVFDvFD/lidHgO99PdYUWriExDWTKLyHMQA7L8Gyd7LYKk+jeW3FE1ZJW0WraBIFiATHFDST/WHnMtyOnlTRZTaq6ItWR3dyV+eklVuXm4sx8cAQEwZb5pXr1tlGtfB17SX5Uprl0K++LndQIqE0PPLJHnAusjrrpa9W4wETYtUi3m4L7qm1vyJ7v4uvcXCjpPhxgBAE2ioNAXWkHAZYIQeCYZEBQyOKITsYPgXGuqiAwnP8PKYxJiQ==",
 "e1_memory_feasibility.py": "eNrFO2tz2ziS31OV/4DgKhvSkWg7mUntqkY7q8ROJhPHdtmema1VdCxKhCTGFMkjKD/O0X/f7gZAgg/ZztxWXaqSUGR3o9EvdKOB/3q2u5b57jRKdkVyxbLbYpkmr58+4ZyfrRMWJCFbiUCuc8HSRLD5Oo7Z4T6bCfh/nYQiZwHbf8M+RG/Zr8FVwJYiyACSHUXJ+sYDKk+fPH0SrbI0L1iQL7Igl6J8MZNX5fMykMs4mpa/v8o0KX+ksnzMK3S5XBdRXP2MFklg/VxPszydCVnhFtGqwv7fKJtHMfye5+mKZUGBwzP98RR+Iuf07TJYLGLhg3yiPE1WIil8uUovhQH+NPrw4ejQ/+Pk7NPH4w899tvp+cXZ4eiz/+7k8+ePFz02W+TpOvPjaBUVEsk+fXJw+H7029GFfzC6GJ0fXrAh48f/OkxEvrjl1dejw9HZ8eEZfj3/8PbCGZ29PxML5/3Hs4uDA9cFyF9Hv4/8k9OLjyfH/u+js4+jt0eH5wB/x+nLxcnJkf58znuM/3rwybdQ6F39xebpk5OTz/7p6OLi8OwYKOXCm6WrDETlPH3C4E8+5Sfr4mT+WazS/PYwz9P823FQRFcCTAVfsSCO0xm8SRNvZx4AZvjtYinAhCLJokSu5/NoFoEYNTzvacLC+/jh+OTs8B2IBN65SlShmLNFVPjT20JIJxdZ2iN1uQODVqzzxNK3N1uK2aWfrotsXThjDsg4z/47+FcWOZFw4YVcptfwas7vGhrbDO5wgA2fWCwApghWNS7CoAikKHrsOs0vDTvrTEGC6OZc3GS7ZwcKTO7eaYSNF+TzOVfwQT5bouyGRIXtMp7m0SICU/bARDWQRvSRLQtSjWRTK/JbzQf+uY4A3Mb10kwkDr+ecpcFUs/JQqCxolzMChjFkmi+Tpw6FP75c6I1AgLx9to0ZRGC4oYKBIlCjMmHFiunH08PwaVQxcP3QSxFg4hb/YzmejKespFZGorGZElCWgUN4WB0GLQZBKVkIvSjBIzrMUL6DwiKDKFLWJbAkNs/I66GyLTY7EluFx55XxBJwX4P4rWgUODMOfg6y6IkEWHlDMQA8MnCVEiWpBD806QIIlxiGIS192ye5qxyEO526EmHbO9fUfYe45HWG2lLcdzBoMGEr0rBIOu6HypXSNf5TPS+21tKNdBiBKEyu0Ue0+lXx5BUeHpCc/Ts2HZSY33rJI6SS2cVSRklCz+9HF7ka+HWgpzNXRWazFtxFYUimQnHDo9htBASLVWvsJ5cBq9+fONowlHiIzp8J+tQL2fpOkGUPfWTRGgJJN8mENQhTAICfbIttiAz3jqDMYWDoA09X6Ehwcj4yQMSUea0rVPz3KEJxfjLIZumaewoYpjDoMHRL6AZ5IXEGTlT/py7DeoiBvoKMk6vRe64bAjU+D9wQN4xYiU/1JaW6VxbODBjobRcxfYU8gFQEWCSPlmeXsvSDbT677hSHh8YSS7FjXpyMJrAcgPRjhwX5gmWIAGS2NhU1iL2fVjQVyAWB3xwHi2shSwWQQ5JiJ8EKzRcIcIe+wp5Hf6bG4uKSoujaYKuFZ0x168lnzCwBANkASijgO/3iuVcxLgCQWQwSyHmnIbcNJhdgsCmkGeA15Fjk/RKYQF/yHmTuRxmnK58/PTnGCCilBHus2IJKd1iyX4sR0Xbr8Tmz+NgUcrTZkO/qrOAFgO4aGu2Chr2NgXhXapXAnz1kVMQNwE8GE5oGJoE5ibw11eMeZhw2yK0+UC++OifH97m/EF7DjJYeq5AUiCxLIbxVdrLRjeLKStShs9SQCEAEYC9S3PR399jsHTKpv5IhNZwZgYvMbFi/bsSZsPoecONaIwBPBtuUf54b/LAPCBTNeNpp1Tah+E8pmW7j9OZp5DsXoM9UIHUh4nAZGCN4pQ6IvmvV5gLGkbQm/w0w+QYGPFkFkeFCXEAOYa/XgSF1Y3D+/9c3fzw1wV3J1gA4K/9NwuT5wXyEt5a+cb8xZ0ZQ2AAC4o055MN60vmjGCZwxXzXC3GfZPN0vrnshcWFd6PmXOn575xWT9kmhym81CxaQbqgWmsggTvz+JASlwqdJ6DYaPHdmBWPeJ5UsUhYCD0C3FT1JaregKrydPaU8HrscXNTGQFOzknrbWxVPFpDUaREGK6PZ5ZcBrc1NceTTBKCrWmuKxcJSIZilm0CmJYKdAp2TGYQTWuqnB8mQQZpHqFQ5WgNIOb1xIrtk0VR6A2gjIKgoYCh6BBPlQLGboUwErVIfgxJ7lP6gkwVxx4q+CGI0UCbcSVDMJKD9SMlS0mL2huGg0/YeKqfxKM5BU6OZv5SlzC2qOKpD8zHHDpr2WwEBWVavAUHBizxjyNaxzIZpxEoCgxq3GZF0DQteTczFfqBgDVVcWgq9wUIaXjNgabRyIOZZmy2P5sqSHGxJMgKaF4RVmJejHen9hWtC2rAe7HGmFvghEBrbGkUFe6Ldthp/yak4AqfZ0LaRzB9hQsNDUFBJslBW/PryQQqeweveC+mXBDCmdikCv40jHGGETIIxHwrk6RoyVpMxl0Mo4AbqPi4RrGV/k5YFbi2gJaplHwaIFs6hsQhueW94cihtjnTAXYG+QHwRzoVLsXch0XDf8n7sEmFYZ3KW7B7thfFKb+aWcPSVREkKMMNcIY8SfjivuJZfKUzQwVqfsAFWMKoi16yk4UsTE+T1jfsKF+N+xLJ0c4KYeDIaJboz1eRnFsntUumXmz3dYMITNtSvD1OzU95VwVb38fbmOurkI1Z2vPJ7gSPmTK0RwzbBRFT8OUEbyAYBWCfGiJwrLCp/0tWL5VRlWsMjMFBetd51EhVKQhiHC9ynBHCcn2GK79STF85bKXjH9JQBICS2+oCId8Xcz7f21QywUkWabkqzhXaUa6cuytKbRwdPGx2T9CKKhzFjDKDr7zFnE6dfhS+iLP/SwKd+irO2mbJtFqbjV11IhL0EVrHwWUGCS3jrXR6EmBdbAqCWtxWRHoCotaaVXdpV/oQtaIQu+E2FWRA3mMb2cBpmLTryEeUzioeZmmrtZ3SkgDqqurFySDkgbJIRHXOJEh71Jkp3xw7lD7Uckgr7yDaFacQWSDmNYtCc0FoFSpO76CQNwIwq0J1JdmU/TP4zQoHKShixVg2coBWVVbTuqJmPNJqO3gHru4zYR+rPLqe4VppVjECKVY9ITKgNdiQdV4I8c6PTv5cHZ4fu6fHp69Ozy+qG9YgxE6P//0bPwl9Cau8yW82++93jg/D754X8KX7s/uc3SmM8A+vxhd/HbeQOb/7fy+gq/ffl/98sdnd/BFvnQQj12+5RgHvM9uI8EEU1uAI0sHvKbHZkGe30INCIWOLAMGNlYAlIFhMFq5VDGG9QM1UAgzSmbxGk2FaETg6H0tH7VzIlVnhWxuiav8bLlOLtlgiOiUKDtvfvzx9Ru3YVeroJiR9zbF5kG0DCNcOIlpiD1EsWlrJllWJkLU6rmHNsA99tNQA8PD/t5eh/cqsTANZ5khMQClTZ2Tcf/1q8GkZis1+VqLbgqhPoXlPVhlqEYdWFUq3WMQ1cAVY1ircXPDaMyK58Va1uoByrDnfBfjyO4doG92FRR3yUjhjZ31KAs16skl5ZywZA/s8mEHhPLqB9IJfOppWYFiKmMklQRx7KjBXL1SzdZ5LpLi/7dg0EzUk7P7EkeNYC/mf6Zy+O5xG7WEiVeygFy7EDp4kxK1HZiwqbSIGtN5kNnb4pAudWyzTUClJY1dNPnGzl01EX4NWvVncTq79KWAhDzE9NJYpAWnDLsMKn4mIF0FIQ7KkWxoMymKQs0FDwfQ320cyhLARKscWkpvIQqHU+Djduqsku37MCBI1jF031NnwVp5Jab+bSMYSN0vpfkD5JYiWmNaW5thBPVXcFvFYRUDMEZMRTzkh/vcOLqWN5gAdoWx3J/jg8Of/zJ4/nnwHFuj9GWxoveK0rhLeROziVwawJAZ8G06nNQxfGIQC+HrAPJUCPzo1Z9PRny7cc75nfkw8F7NN895M0IkGL70LjjSK6OM4e4+BU08ApfNYtYKFZpeFfvcnvrPbH2p1u1UR72OHZp1AvYUweRjwWsUa9NUb3cpZu7svKbZ4pEDbsbJcvRUexdrfKcVvJmwOxLuBjcZKZf5xk6N3NhdXQMb+MhtOrQmgysAJE5EC+5F3Q1eTFxEVNLUkFqY7oZbBj6P13JJbZ1euYtW1Yp0zII8zDE1hyk3VIu5WsTQJNN1gTnlVbWIUZqUg7SGr/cgo+9qGNanSx6B3XR4MQ/l0HHrZem44ZGqvKSafYtPlqtoXlTOBStyWsCqPDO2RN3ALI3ISjWwaUfBDPJ1prCr9FCv9ZaHTTGlbuTejtOqbNxGEw/zK1su2zFMBVOJbBsqfKWtBF+nHlT5xdXQHbl/j02hTBQ5vtqnYTSu7SU6kNe7zKdEtK5aoy3d1+ETSD6vw6GyGd0gpsSy3R8+vzg4+e2CDGmojMkYg3mw5uw2twdMJ0/lKEOuXYyrrhFWkEMzEDxb6Jg0lf3Fdi1CRlQrvpse0ewPUxqMntWRalLKh4HVV3z2GOUFwEAqPXz/g2Nx2cO3fxyf/DI6/tAR+1TGt6ULbHVoarslUM9t84UGbe3ZtYxSdTCvWb90rL8PDeAWRup+xBE4xBMx/Lv4prL4GkerPHbLgG0HrVdENS9qQm9ZYlS0xWDzQEpvK68mqAeHaJgx0d8Capyb9m/srZs7ruuBAbMcYGdHUduYnZytk/wOM6/66FsSnXp834JdC78or5ftJaSNqfKkWIjM2fN+tDN5Vf5D9T9Ngzz8aEyvuelTN0nrJ29sT8a37f0idIZuv2tHDvMH/Bh3E+vOrY4JeucfP3z6eHTUISE9nVOFdJSml+us2WGqRZdAyvYX/7sizl6Dj/JIUXn0pkJVFP0i9cVNVOA3xxqpVmHVUlwr8lQOUkH7/wHnJe35j3VbjG1dpXg3vXFnAYKpCInWy9fgRTdYb6viettSZdO0hnqUd6PAwVzRuW0ytovb6ZMZ0nYVUBSqbNih4WjepXcIvnvVrpe1GIP7rIAJ3JPzlVEP+4+m+lMHUdck7ditqFI83b24PyPsbckYdQ1n9hiU9gQem8X1KMXOeH2rWqd7VS8AexXJrdPu5amy02oi7Ll46qT9udZX2FO7y2VDEKohNcFGuePW82BsWTTaILw2GTCP1uQ6amHTe1Jj4jkeerAhQZfIK4BGySwKgRS1FTSBcr6Dkre2ISCrw0akoyzCkmtVPdubv12b5SbNbZwDaEioa9dhQgcfgHK5NW1F/UGr12Q8TXc3DaB1rKNtP7hbbuto/Fj5Te4fnywnnUpYDMsFilhoO9MDhIhndQzapqMk/mzIHrnB9MAoMNcUY1EhfHW2g3cfFOpANYihdV5iFUSJE+SLqyEV9JoCHd/P0SP1UX5vlC/WeCL+lL44oZCzPKLTLUPfD9OZ77s2qheEoR9oHIf3++aAaN8cEMVtl9tMDE91s+1/1lEuQvso5BZKqzTofw3y/xO+Pl33XXhGbT3c7QhAuMPG2f770fVJmw50ffj/fnQ8lWSmDC5TUdm/Hw+N0hqTjJQ/ICCdmtYlbCjQtjJX5iV3Deyu2De7S3MRyGgaxVFxS/VxefRyy3C6vumbXbZqkg8INJ0FGOiDGVkhWDosUH4BeoSXSxFnQ35Em20QzaJQtbRgHAmRgn2imx4PyMHkKX2TKnfK//Xe/VSMi/crF3/sDIGBr4DbX0RFHyv+qNxXByhJvWjCo/8QU5InV6fr8JXXyvixVbNX6xcQFaGOxHXMm63WsmBTwbJURnj/wz7Bh/k6jUP6KA/h1u/KYGsvjPJam7U+LF5CwkPiWjXs3elvSEdMIS3vMepHQ8anta76xvRUKddmihjShmW2b1uVdifQA7JpGOuDksmFh1epqIPm5Hy81/9b0J+P+u8nd29+2KD1IhMQknwVktx7xq4CVzksdhNzYCWH1c8cAE/n9FofN/58MsIjvdYxReOyKrrD4Pq3h7tb8ZUwiVH5HvgAg/RWl6hB9UOqvU1Mc2XROMFOWcSwQq+flaDjBJaQSjAihadfQLV0RMG82CqRd8s0hcQ2oBZrOStqDBUpLN+C1nTcM8bGmT4yXyqIxlBzcpsndOwkrawJgCCMTzW/3Y2pLoshkApKpQkqG1WNQnW7rIYsZ2mG55J488pdDcosnb6OAYPWxTO7eaNihr+g+kPDmzjQ+GLjmdVtwO44Hp0xWOYGRa1fpZcyDaJ/1iYmKFGmz3S4vJlN4/k/vEJYNon23+hybmfndQsad2sH2CAgpSF7m93qSIutjTJz9TFzbeI0Elsbs7m12znmlv3fx/XyGlXdg328JvzjoMrCs+NbR/3QAVQPiEaLjdf1ouJRW1vNQyh4hBmrQqzA4zQIpVNd/qMhK7vXKaO6+NA8y1672gGhroxrEFQx8pmwhgd9Z4Udqcp2AoI94jRT64YNBjNfX8tQGHTTi3JL176y0biqRmSeDZvB39xEaQzbPraug3p1x0qd96itBue/jPpIk7vtokCLRg9LxYFiyhbkFd510bedrpcRrF/ELn6oz4dA26eQulmnPtt1oNieQ2kU4jH7gqlMFQM3zuJK5NE80svXdB3FIfv14FPnXMpWCHDbcePFDmK9eryywpN198Xaw0muyiMdVW+1fowjlZ5eAbyoECtcv0AiAGMupXTe2920pwHuCOlr6FvXBvwrWG6wY6rKN0m1sNN9EfgvNifq3GiHrIg4yFYCdUVzy13auhbLQ/8l6n0NJjxLo9MDHTSoS2n6TlaTqX3vq7xRW7uEuy0a1HVr7SrZcy6rbbNJ17pEp6/t1Y2aaLfL81YS+dhTI8bb2x8fcpk5P7VujqndhDDCnqJ1pMyQZXdbRtlUx/lqbqT2aHH7xr677nQ5mg1A1qNwra3V7+lsd64qqs3dXbbUzyDe14To3jZpdiHMYUZ9rcQ+wNizHeO8fNTfzO3Ut0GoL6j2mDkRSYuHaF1T6WqhEhR2jPF/vAft0JO79f6o03Ub3L3/OumjV+do3iG22p5RrV9NdxL4GPLP06PDi8ODCWauwBU4Z4iHop6zb+xOE3zRkbm8mAx6myrM27VxdXbCtQ99QCZm6CkGX0w2kKya6Wy6MPXpk71HzE4l6ftqYwzAfboP5/sE5vu4Teb7RgbKR89vJYT8Q0i4HNpFQ839G9HQr6A=",
 "sgbr_c01_e1_integrated.py": "eNqdO2tz27aW3zvT/4ByJmsqFSnRsuTHXO3WiZ0mbR17bCe7t46WQ5GgxFgiWT78iK/2t+85BwAJUnSSXn+wRBA4L5w3IMMwLrLkLsqjJO6zO55F4SPz4oBlZcyKJWd5FC9W3IqTgs+T5Jb97i3gmd0n2W24Su5twzB+/OHHH6J1mmQF8/O76nvgFbyI1rwaWHr5chXNq+fPeRJXD2uvWFYPSV59TVdeESbZuhrIaoD5siyiVf0YLWJPeyznaZb4PK+BFXydhtGqhtCgr8xWQJ6d8b9KnhfV8JdIrgmzZM1SoBNmMfnygsiu5nLHXfN1kj26IffyaB6togLkmcOLas4tSdDl8V2UJfGax4Wbr5NbjtP8oYPQfvzh8vTN6eXp+9en7tn5sXv19nh3PGFTZhzO+Wi0Ox/uj8Z7vjPaHe1zLzj0Dg6CvZHjHYzHXuA5e8PhaHQ4ns/3h6ODCQ/9cTieeP7erjOBzXpzfO3+dnzpvj8+O0WQ68Szdoe7I3u4ZzvW1fvji6u359fWZy+z7qNiaQU85XHAYz/iuQ2jAOK3k9/dk3eXp6+vzy//iTA+B7eW49hDe/fw532YcHb88fR9c4qXev6SW2vvjsfWyD60HQchHX8E9k7/gHnvzt+778+viaaz09+PmQOT9lkk9DCN4pgHDKTBQB1uGe5SlPGceQVbgawL9pt35zEgwhASRBoVi+fAATw7zgcLKHUfJnvuKorLB3eZFHmaFK6k3d23Cy+zF18kjx8u/4DlobEsijQ/GgwWII9ybvvJeuAFSVpE5XoAOlVmUew41jyKvQyENMg4UgRfguQ+XiVeMNDk82L31f7gSVG3kZjqHR75B+HufDwKAme07wewf+O9cG8YhIeTMBjCXk6CcH/X39879IeTvckhBw2Yw86Oh6O5v++HlfQV8x2CR1JrTsX0Nq8ZTxObFtkCgp1kiwEN7A7wqxgVIwMdyUAgGTzVdGwqNMDo2NkFTOaPPzD4M+Z+yPdCPhqO/cPJrudzVNX9+Tj0973hwfxgPxh78+Bgvs8PDoe7+8Dr2BlODkOPzzkfO5P50JCQAh4cHE6cvb1xMAkOHe47hwfjAEGHB854tDs5ONgD6R563uhwd380ChxvxP3D8cQ/PNw/ACg9oTkBDxls8RrcoOlli7zP/PugT74iKYupszvsHQmUoH/lqgBmal9jg+OUrOFftX4qYPCHYnqdlbzP8iJAaNrKi3cXp/16KUzgWaZPuLo+Of9wXVMiPwH+kvu30zfeKucSQE98RKGkEbxaUWaxnwT8qEaReVHO2WUZI6TTLEsyMzSekOab4WzD+ENUgM09bYHYfIrVoGDjxhoPh8Oj2cboKcngXNaYVEs3iBbgYU30pX3mrRZJBoa1nhr50gMrMJR40f2Qh6WZPTsBMzaNbG700FfmRca9tc6NwCnDjI1O25WYxFwNV89e8gf5Utv2MgY9vnWLJFmZGOL6QGrG/QI8eh/jQx9kksIzD3RYOrkqxNjXHN29lz2eKAhmmvEwepgatvILlkEIpoiKeCrUoiNdD7wFBGHQMhJFNaWna5m/jO44KqKcPEBq7SxPIQCZxgDwOL0bZ1YvIWKbEc+GRxIxMaqUbDIkymAn0yQG/VLI5G7cP7cbRDoFaPCX6SPKJJl/Nms4YoXGRVQphkShy5j9NK1k38LSqcWv0STycs3WUQ65hb/EsMGeFPGxt+aVsuKfMvgbA/wiiMuwHr6EBlGpyOnh6Gs5JgXdm/WVzDVgZr0Nlf70QMyIlfSq8aJWP8oDpOJBpMV/d55SL7XQoFk2pFpGzy5jCGS3JjCJiZqb3JJ3kaSAsoFFoFoYv0BM8gpI8hg4kzn/FP/iFUUWzcuCswcWl2vI/PzG6JfO0UdtFBI871NsNHH9DMgM+3MSxehKok3/KWIv2C59voRPWEA7EWFgz7x4wU2nzw57vRaTRKbtZWEIXN6DDnAXnacp8YAhoicCnqdGWYTWgdrLwstv6+hCceH0zluVkIxekJ6DlnirS74ARcSkl1k5M48BzRtQ0CtSSWaFrEbfY4YGy1oxc80LzxZTAf+vmRdEAPVVkuSgm9cZh6TEKpj1ByQwQ2at2f4YkTjDbkDnX7xX3kLOwNECIdhv3p1dn5zg8EdIKrzY55c8KH3cwis06tcokgwZaEBdsDFA8dnQHjrM4r0es/4E02dWxA6QL/gXsFqBGsFilaCTqQwBtZz0D9X+f9YP4JnXZBh+Km0AVBRfQgJpnyTXIHgYR/mDTQgd1nxIa3sRsA0Im5sLA89uLDmsThsgRxTze7AEPjWMDgjd4SK5z4HfVQQOBwDZJ5FfXEI2zTMZLnp1EF0BBpxPbshhoL9Q/HgFjUGovDEg4ctitHgulA03JorBDcDG5caM1h18K/KeSv/GkphTpvv66iOSKXjn0WJZMJkLwKQKfJ8tkgKiNNDSjr9PRurhZOOIiazDqNa7NXlH7ADe5JzmgTUaS+6l8NVQW46bCs/4sdG8FTg5Dkqc5/CpvBQMrlMQq6r+7OpLnNyb1QP++wJs2mXh9+woT7DGA4HieA5SgFyA+0kc5IqhFHJssH7j5okwbGbsSWLeAIXhqsyXyvUp+jCKu1DSopnnZp2zCcng2yMsuChju8FHhOMtYLPQQdUD6Kk0f2Ia6WOxTGKyBUvCN8AQTNJqGt4a9beH13dxa1Baoy5iqCT8woXywfQxwVZM0EOA1opfwCAaFZmaEwcRChy1/EaskAkAUYTZgojP98vIXyo6a6WnZ01na4BQEWBNaFJCQtMwuuXJ6o6bPTv1INIV8qPXJsYGQ8e1OdTCXIIwBmWeDSAVGXy+W4NFL1bJ3DReOs5Lo6esEPdkmaw57kYN7agRwVPw4RlKRe0qeilaBC4Dap6BZFEX+0yL2/j6W8ufXw0iMxURNwYmzy4my8aMTacEemtw2ExkKJXD3gu3c45Jh5kZn+Zisxznkw3qUsMHv5qWRYMAHcaOpJAZUHfuoNAEBdWydhIlk2fgtt7KShVUFwYKQkiFYniFDs14OJi4kz3j637NeJ+wMvfmK45aCozAVpZxYLNr0FzqOEWgyteijGYqNa4LfIHE1hRT6DIoErju2rCfTwWFsSByD/ykWL1hoQeCDCQFPrlsm13ydQKJdFTIjQA1qDDrxQEB6TfNTjxCHS2+iJIeHausahr+R/Kgibpl81RKN61+DmJbNcy+3WeReoqupZKWXAVO1sUkvCkvQYycIsYJ8ZZzQJD1Dog5Uc7egw9v1SrcNN5haIHSAuLhGc2kAOalhbXghZ51F41KhyA0K+kbQ60CQZYp2jw5zb/+Mmaq7hWRTeUao+Gw9/0gI0EpwXzE/8TZc7AnDdj8wedpwczzK9K0vo7oqvpK79rWJuUkCQGrLlcBiyGKS3qE2Gz2AbN66n/5sp6xKosRklUFjS7V799BVPLK29FQ07lte6xt5SEPTmuft5OWnqoBshW9LySsBb60rKVSUGUkqWpaNy0EAiXy3Y6addSDtzhH9+fiZQKBSXRkbwxqSr49PztFnkWmG9z2tuddHF+/pSlQ6VBCK0Le5gkmYcci5yl9Vwt2cMHObGPU9Kj85NnEu9qLvjD6ng2vo9TUePKfBdLktB34uiEK1QwNaqcKSULFzZ50am3qK2C+De4XW0Vqu0DHavnrHuzrwoNlf1N2BFvjG0NmOcd4+eDMP93cDK3/mr28YdYA/v9i/d8MTVuEz1o8gLVbIC2JZCKcSCwqMMvRo0/5S9P+uQfAGzRpDTi5XHpLLB7IZOT4jTNT6LQkCsMrbN83IqtwAZAKke8QYidvoY5rdnJWKXPlItQOi9XKn8AWNxh4do+lSaI+9HHD++xJq0BJTTCLgHpBWU5fDvMH7pcFZgLqpdRyXbfglf7Yb8H2W/OqZ+W7t7HIfW5wB68azxKNlnqXqagCXfK84P2bniZPysznWig21II68tKm0LyuXEW+Wd8GUWZ2dqIWkYxR4tOimCeWVVURz8Fq9JxVrao7VWIBDEGSY2EKQK/xP5gervtwcXV9eXp85r4+Pzt7d73536cnJCIqNpsqv8QkUCBrxoSfmjFhi/iQF/5SlEoBRMvl1EEz1JA2nyQJFZ9dcd3zi9Jb6c5OIcv4nQV+JOdVbRaFj9/JZyXZpvED3xLfT9MuSr+7ptcOsIBMil2sk6q+yI/Zk8C7qex2i12lmUq8hUeiFuvaqrK1OojCUKyERFtkQ29Pj0/E0JamSatXuyJw1NaivIj72ctEt/6oYQY49LUMlM4uZAevkRNS5iiPYO0/oxTbdAIBNnUAW3cd84RdqYHoStn+ystzQ3SqBvSAtGb5ABtwg6928uTaDfsHhrUCW17UO6aukSpNZSKoqHzlBZLQZ/lUYpuX0Spwk8zNeJlzs5JuVkQhiDiXDhZTqVwJvcp1PCx1q6nggvSTXeUfoJqJscvTmoqicAm7jcfwhiowkJwowCZp8QhLdM9eO0SyGfCggh6Qq1jX8uJE9E3Tt890dy5Wtd2yXNYcntX+eVtFqmYIcmJj8ZibNeMQU71AtBd7PeHGqleVQjIO28KeNo39rBP7j96q5FuJfIVXraNNbDafwc5lm02EjwAPfPxCP7sBU4eMX762oRwwb/ljD5NtbCVyanzASF8+RnFrm+yo4Ou8UkYFs2GR2KkVyX6NpapLEZU8eMF57QNENF/iTGNdtIqfUShs67YMGIZa50WiD7FlvN86T1V/jQTuFfmsuEiruGf4K+7F7QLvJL+N0mvgM58WUNcZ/W24OEsceqPeBolv4xo5HV+uoAJ9oMFXyKx4M+uAhGe+MkkQ0jGqM1/Be/87Tnidg+Gw63hX/TW7T1876n02QGFAon1TDRHRbIYQ33Xwa7MrDoYC+7lplJwIgTZO53hQeBnW2Z2uSdOuhqoSpN73UI79o5p6lQWDKIMSSPBU0yn0CvKV2Psh4E3KKRsm7wGuHXl38zIMowfTwMsudrFOG7ObJ5qC2r4EsgUVJJeuPLD92q5UHa8yx+BG2eFMbCG6EPJGWyaJAT5a5U23/PJl0xtgix+iY5Vg0wmNxHBUYcex6At3548F9f+RUyAZXCR8uPhKd9QZDzkUZz53K0Bd95Mox4Zsj+dux4qa8Wn36oaH546dgxlCdIijkO4KVF67rwTRrkfAUT+9fClfUjZYiqMP+qKl9rJn4qaYiCVlbiossEqofK4UEGM9Stw0tPtaImQChs7rXt94i2qjnWoICrBDXxEB5oJoyfHTF3D5gg7sMLen9araYqZn6gS22Zn7++cyoEIhjpjGi3++WL8Irl+8fXH24sp+Ef5pdF4/kNITB7iCBhcceI4ugCjYWiTrH1H95/rptWr6Y3hDEXQwRezCa3W0rggZiFEST7s5GRoXSuh4x5G2Q5Q22R2dqlWXBKjiVTpz72UxxLcAI2cujo7dVQR5EBS7ZZaTC8joXoj03a48wNIS4Y51z/dQr+T5tMCJ88r4FvYqttnxapXc1xffRvaYCWAZX3sRHUOi4E4dCvki2uDliSQrGq5PWI6yIrV02kmnxehszl4n4GSTOPIhabIU0z2Q+GgyHGqXjhS0fyB5R9ub8N/ivPSIJfHqESONXHBk74YbyU4SitZnEofRosxgd/KmUMQqGzlde49sjqZSQEgtUyBKPzFo7QlKE2MF9WNQRu33gh32n9Oak+0dqlhoEQnUSHi4QXcccQF92QJ0C/dFXqHd3hela6CXbhqlHJsvdOINWPoY2D5jW02k3332ckvVpsgPTaTbDS4JA7LG6WTYlep0bLOE0HwjYrp6JXZcPq0SH8BTZqIdAOMEUKMtfYnC6q1qhlGkk4PfspPGjmk5IY6vRX0ZYq+EdxloD/tuXYChsBviO9FB2+ZbpsNbdX5dE4CtlmkKWuyxNMmjAj2QUlSCRARH4BZVfQfoSDPwnAm8Iy90Xd3avopE8++qMS78OuUXEhmrkFE9Ej/qCq20eV2CtwEjU1zadUtQuP0pk9qKAUAOVg1bFbYaUUINy1l1RdGYVQ0b2vF3E5udL+aZRS/kJLoi4kJwacLyh464b23U58rVXSw66teyAa0QlycZFVRdrtX6jmiGVwYhPreuZn1PCiIzoTrvAEb06L+dkkhRQwaEd7Na9bvyH4uoqEv4tlMxlN9xMR0sMTfESA4MgZz1gslQLWYhTLe6Z9IuU4xoEScZl+0BvDQNBnaHd5rmK8o95UUAyPioVX1+QTfBPx5fvjt+9cfpFfsP7djAhkIYS91msqgnSor5m2/gnWnbp8FPk9TU0oZi6yYm+H26Wohq9/riQ59dUgtzkSVlKqw9F+Yu6k60HmxoNI7nahLBW0XeyhVT8DCkdWfla4sETlqFbUTxKFxXY6Wguun/IQmiOyzsqTrZF5daFG6zt2H/QgaZF5JThRiNl6BAVDlWLS6EX/XGHPZottHGIaUCEgLrSfGAQTGx02RiZ3azI1Nl4XqpLtmZbZgoUGrArcsaen8M5NA+FewSnzrSagk9r+fS3T8Xb0FjWt5kyrAseYNCNNc1E6RWhOVhbmbJVM2SYcAirtr9BpiNPFiCIpyGNgWAO34UoS2dNepnisCtlLhmQF3UAVQ00Wg2DL4RaJ8HK+/w3GxnFYCpwbRFUNVRREdc7m8vEWHS6M5DWkJsXcBBS8CEzayJ/a5ewmtr6JAe+EsvkpdkZUNEtjy0nd7IbK3lFz7ScYM6sNf6/bIpotr+jX5Co58O+vbcsdMWtgvllQkb/S5GNDqMho1AUbxOMBT8W63mLgPCVhZaDXxup/VIBzZbnnZE9b1Ddw0SsHj5PBMp3w41TnbQbUD5b+FvX55omugX7Mw2LVWlJC9B9M+1GGbddyx+8zLZSmF44gHujdEPuVBqSwjMSRaBYQjx4J2kSNSFENCTDFIomublSyxG0LdX1y+U4Lp1AdQrVntDukW3Ny28zCl+7FVg0tPtoB5jWFVEvkxT6A4CXQyvcg/9dngXiO7YjIAwBela0Q77OJdSJbm20Rj/nT/OEy8L3qli6+i7QWr1mdHKS5/pveuGqnfiv3rdhn5yh986KLPFNSKzRR/eOC0ggAqjx1vEuH5KTp5g9jrphRAILl/PEpK5bCU0DiT0XK19IqG/e+ZIorMnpq2j1pfCDLXhS8XtptJOZaKkjkIJxeaCDTa8W3Xj+t9ueair+Nj93Upan++ZqdS4HXghQCi3aGlnns3DbcvCHzJ+ph9vaK1PMSxdRF+4EOUw+g0U8vJu4/aNVdUn4oVgSbxC6ebi3PT9n6cxzxaPRhMiXQrn9GuSq19fXZvHl28u+cJ88+7y+uSk1zNk6CNtMxzxqH7jYtXXs2F85AyHk8mkBV9VipYq3iSVWxWkIFjm+hYUAJYsALa6CgL+7HsaJ0fNX7lVOQFgkqssdYdbCq+lJdrVgnYW87XcRfXHUPnQxZ46oMBqBzCeLOb4g44efEXZMnztTNiv0StQxIdoXa4Z3W+vDANbLpVfckHJ4evaxF8TY6M8zLAc2Cqit3ygglb1PREKum76VbItPkz5dPXu1+vTSygctvB2Fh71AQwA5DLDodzmOSf0dawNGnvNAr3ptoTCNzxWc3rlTbeqy2m7thQZCCUYU5lm6D+CVhCnujvUarEt9ydo61ektK43gV48qVeQV5CTx5ICvJ1Yuf1jRSXlH3/4f78rmp8=",
 "summarize_e1.py": "eNq9OsuO40hyd31FmnMw2atiiS+J0q4W6NmqmVl4prtRXV4blgWCElMSuyhSzqTq4eoCfPDFZ/sTDPi+nzBnwx+xX+KIyORLVPX0LgwXuvnIjHdERkYk9c1fXR6luFyl+SXP79nhqdwVuTcwDOPjcb+PRSo5u3bYmmeZZBtR7Jlci/RQyktxzKN9XIr00T48sXgbp7ksWbnj7BAfuGBxntBbfASSQv41E/EDE1wes1Lag8H7Y3k4lrIauZTE7uny2okqoLW8Z2aRcyaKB4Yk9xxIJeyRJXEZS15axOQMhTJeZdzeJ/bghm+44Pmas/s4O3I5Y7fxyvaYeeMO2cF2AkUDB31mPsRZxsp0z2luYrFi02g0ZKWIc1R/xROW5oOkWMtL/ghTgJGX0SGLc2DKfv4vZ2Qj+ewoUe4LyQFhk+ZxhqpIwGWAdnkoZBntinWU8we2KQQbaw3BPuCAwSDdHwpRslhsD7GQvHoHu1SPn2SRV8/gjF31LMu4TGWZrmU9clwdRLHmUg7IjQeAztIV09MfEHnw4ffv2JwZm7G7max8bxI4Lh+5o00wWfurdTLywtBf+xPX50kcrBxjcPX29u3H69uPgLUwNiLliTFkxg/FUfLwR3yMV3EGLsTHVXpHd57x+7iEkMCXn3gpilsRbzbpmuDTjIsih7kBa/0Z3yHtfZz/yGMEq16/F91XucXX3RNY/S0+XXEYTb7Dx3f/cJ1zsX0yloOPP7y/uf1zRbZrcWs5bcW7lqmWhuRoFPhFgb5hplo0eYyxty7yTbplWbziGYRRJ0yAL4NQeQcSWoOfrm9/eH9F1idupvH+n2Pz43ZVWsiBXr7/9tb87vc3t1dXVmfQMqyhRvooDg3Sx5sPPSTiVkNvVyYQUcAIiBTbHKrBNgfAeSs2zfTbm+9u+LaPBuMnaCBcM43CfUGusgbUQKcwbx+3K4R5+/fffytOJ0FkkKmyUUOhmmioiI0G7KrRjLRUeP837yis6KENqWbaTtBEW0qeiojDF1dXFGUEcQrww+0NhfNtM4Hh9YGiS6U+zHrM3PM4ZzcuK+5hImCYoqSFoZcd9zmlqHptFyKBWB/cvv3Wg0h7Jk6tQJuxxcgOp0M2sscjvPpjvE49vE5oxPXpmcbD8Px1TPDTyVKp0gpKYjCe4HRAJHxiFgZElNC8ccNAERoTy7EaIfiA4KdhzaCK45YCAYnihy0FHFKARiZBT/RJc1W8uvQp5on+xG1kCIhySJQDNU7P47DRSPFVkiuNgkmfPi2Otn2CRv7JtKHsOS1reI2rtMNazms5ANcT0VYOdYjqSOlNsA7gXcAQMfXChpTvNleXBHNpfBpUxGkhKsOPiAjaBG4O4YR+RfGiehk7LduMWtZS6DhxMbEn3K3to1etYqKsrMgqn7mNJxxSYOw2XlG2U1cFqWPOqRVQOUDZPmioa9srToTjjhvPKC8p+HFLL+Wl6bi2PeWGtuhhcw1bq0w5Vi0LJbq+Ekw4bpRsEVeZhqirORWEvlpQbmMk12kCL2iZQV9buA31KkUpy7REdcbNGlIJYdSabce4XoWT1sr2K/KY4FpGV9OO31jBpfGRcobfGFqRUzHpBU1Mjl0g/dLJkT5VfzpHtmpCyWFn/oo06b+SJoPQBi3diQ3sfRukcnwbLOyC+YbM82yQ33Xx6kxce4I3DxHwBmI7ts/hzYfiEuam9oh7Z3NlSIS8KUJNbDCxO0Ve/hiJTKZIMQgIRJWpXuAR/3FA0tkhcnGQPL653D+XMD0ILoByEdPBS4CaTJHdlMYcfAwdJDlRokyRZwDao+IBIYFeLSVa+dJTlvJJTLKBQ2gOmg7shTYK8NkZhWpCzY+IVTgK8E1rNLanXSZV0vQVDgnkkToOYrtkbZfIu2Ni7jqonaNZgtag28RzCRFsGaKhvI6hKHMG5F5n2mgxUv4k55JXgAGCeBMyGvgIxfC1vx10gTceI2DYNlWVPZ2QYgHVRk08Mv8E2UHI4MiUYg3kmKAEE6KnPKtvYFDuo/em6s3jQT9/gpDjyq/45KvQctG2xB6ZKI5jiogxPXshmU3bsCN/nTw9ihOIehfJBpWLa0eH5IkRjvhIRmk3VbE0paEJsXdQu176DMY455EfyLYueQCuQNKnYeVGtCSqN6YQcECNUMusghXXSSeO6hwKGoRke4pnkpTix9G+R0xiGpIvpi6Oh2Q4PyQJJiHi+uju80mUUhblOUpb3QvZNqwvE1SUJiaUX4KTxEn2Jos1+Pri6wvlnuYCqpF9HZW6KFsOBgnf1J2JeTzIUvB4P6w68yEVlENqV6wZCVCKp1ndEpX8sYQk2TSl9nrH13dRQacCZqf3WxjblAr7i9/BFRjV7CARG3JXPMDwxniG5vVldtpXXz5riV6iZxTp5fIZZXrBwwVj2W0yZQkJXMxbQl1d/+Hd3/7445Dknd+KI/RcCMkf1/zQ7qnt38EuwZMP6u1aiEI0ygpeHkVO5TiN0RnAnGXQoJsghn2VrssbHgNzE/nY8pClZZbmXJqWpRhqEpusiEsT8RcXznJhxMmnoyx5AvsRxw465XmJZxYJL7nYp3lcpkVuLC2WbhRXnkmuBFEeFHF+J011NDJku3S74yJa8RLQtdtoV0NXFQIY1aB3/GmexftVErO7+xn8X0BlBGJCNyH5vEtIa4CnNEOW4t74ApFGow877GdT9huW8dwkXlZjuE8Am9ZvCvYT+xVzOvB0gEOPC5pcgihsPtdDKb7OOn4GqDlz6iE8ebnDnRyMseVmOlQ8rC6Skn+hiN4tF6MlMGFmCqCfLHbJXMSpMVDLT/WI9p4ioS2/2ZfKmLBmUghwOXc1R/AVTbBUkqt6gWT86V/+3ahA45X2n8V+C1pxvwcOS4MAZpBgXozB2ZlnJcPLBgCUfPs4zc1YbO/n1EYqqnQMhdFQHUnZb8X2iIdfH2jGTLg6HoSwm0dRAmswslqYdpwkUaxRTOPiolrJsILLpwOf4zEUhtE/HVPBk9aKewVfn/110UH8GEbpzTQqEL2SABsXn6ZHN6QoSVdrQDDqtBPjdFAFCB6XYYwgmLQ1TXC7IeIHA+/XjmHZ26xYmcabyzeXmGvmby4FVIwisfGYzrDansFhjBGYsGFRJ9JEDkA4TiJMAqYWV3tZ09ly0BrP9o6gD8a4sS72hwxWe2J0w5VUsCHvaWOYpiKxMNQRYwQL2KAVq0Z1loRcMWSLpWXHhwPPE42kzYIZZMiwKI6Eqx+oOp4DypBWdcdiUE5H6kBLn2TR4RUYUR9bzTpLMM0T/lhvIHRGmoOfRVxys6qxe2sSpZNNdlJao5VMzbI6KSatOmlLzJhYGOgnY4knaiOrQxuMnhdlxaLLlwxc5GWaH3lXIBdtAWSFC0RRK0GJRRFZdmArywH0vojJkJFuMH4R95ij3BwjSM9DPqqV0Xmjg5HwDAxBnjpVE73S148EUGHaSNEHU26oztjnTUlA6+RMXVAFnJJVVwhnyVLYV5RBJ3RHNx+e/iklq9jFzFhxI39cNPSsLktq9OatU3N7g0OmcLtw6pB2zvAobNEE+HJBwbvsWxxgifaFQu0AwFaZwM6PB+77uFzvjCqdE6bFfjPHPtZTe7YBmapM4+wskDPSQPtUKlKnPm5kpaxB/QPtm9Vyw9H6QLpv4Zas+WVs9IwHCWFhtld8te6W2gJ9DAz413F6rkDwrjMwHVW+fu5JrPMclLxtFu3sN6vSUp38ZnWY9skJWHOIAmWHXgxYfUI4RvghTBwzDrPVujyH70aoBwDhbUgDUHBGMsYEDsMtjbEQvcfgQ9cRR3j8LZQ95GXsXc7SV4qW8cqrtEbdME6ie1mN0zuM42uk3QrD+ukM4X38GEHARTWhurxGbrSMQaf4UUWkJJl1tqmrzTNkayp08hzhPhYLooUKa1pn8ChPUgDJrtFaYTLUYLVBfALGk5kzq/Y1JgKrZ1IYUV5lBpv/1xN+3D8CoXbOxhFIg8b6cIz2Beh9Mt+ML19xDmwZ66PAtPakPQFbSnuwv5ucM+yBx3eRkDLapqsIqGhaSKyeWj2VXFbbZZ8qlsFv3ngn4fmiywfo7VTl2Kmf9FdbowKx93dJKkwMhhyKYqwAh9Bvgd2j4q5VEOqOpklVDylUaCYyoWqs/RUZKrMC0oRpYKcITSH2VnMDnsE8RZLm27lxLDcXIdRUELU7yIunO+KDSEvK/FW/9nc0YCpY3MV4lqDv5ZzaOmrRRsuTbUZRsem2Uw3f6wBIguho833DrlRykqzIsRla7xi2W0/VZ/FyF5cgvIS8/qR7DRZnsqCxcgc7qM5ukKckfpAG98X320v4fwNtoD1QqZk+QGOhsCcI2Lsj+Ncq22gHyp9ME7d1Cyf0DkDwSftM1FJbYoIf8YFi0odAYhVP3I9gRb1CeK8GCFTTpR6ZvuN+gz9SqKLKrNvim5//qOpU9ZmLUs2v9S6uPoYBTRVqOy65pO9yJ5+eoUuqDV9V28p2UL89Y7KqhHqpxJupcVQbBqEfdzybdTdNZoDZlcbK/OgTcGz2BK4q4FH7Cgpu5FUchax/UaF/X5Gld5y+QBvKGiqiyBw/qb1vCR3ogj52Lyp725SazMRa9n1B4CgXfeBWYhltS/8KaX9mBsAZDO72pwKaRMXXUoOI+pkALi4u4OENpXMNomjhmUNECkGHtZ9BRdrz70sDWJVcX4SsVUk69SF24T3FO00HRJrQrf+dNBcYeXU5Q1G4tHosIVtTGrJhke6ledqTVNot9su6i/pFplRQ7mHj+Mv4kZH6DE8pNZjUKEX6FK7VElAjNlS2/MX2S/tQG0y1XWQ0q9+KEq9KPDrAwJVPFHS7oqqFjfGMJyM4AUvHpJeWcV4so9MXJ71+ADEjXFtnasgT13bD5kyl/xqdSqAvLao+SUWspSCSr5WsGaKOPQ8rxH5/0go2a2Y7G6R2HqoKEQXWYoFmn50RVLNtDpvqPFC7sZcKFmDYN42/h0hqSJSWVX7Qu1kroWDKoPxdf2ljpvxi2va/Jm1Dl7OSRXaEbK3baihLUW7cPnexSB6AAOXUVq30a6YLUUbVn1QZ2cRfdrE/BJZ9nlHfDn9pBq6zaI9FJ6eamFTrVUmnlcv/7xWP3vo/X/ND0KS17H217Gm0vfL/vDA8jT3wb1o+0dlLq4alH+Zt8EzoKzqeZftAYjmou2wk3CpJCyHR3Lom/yq6tUCKWKMzrQL6ZrKnmvyV/ukLFLprbmN8qH6aqA1yL8/+WJOZpx9Vhs0vFa3ZaWHTqpvoi4t8Ydg7VwsrGaI52Of/+Y+b//7Xn//4Gbd2stPMdvmLresN3Y6eHMLWjmpMXAHiMVqnpaXj+WqW4rQHMcQmpjqjb4yzMVRpiAHUCGKCxBj3n9mf/u0/1cEMHbUM9bszYvqIBkzyXKHVCrUak+rHqdCVUKGvznuNf8x19JIkFLIwdKZHUTEs0ryPBAE+gDiMqA2NIjrViSI8yo8ifaojYvxJ78cnqI/3149padJBP2D+L9M2Gxg=",
 "run_matrix.py": "eNqdWuty20ay/s+nmEUqFcAhIVG2U7s8ZqoUW06cSJZWUrJby8NCgeSQhIULFwPoEhb/n/Oa50nO1z0zwECilXVclgQMerp7unv6NvPVXw5qVR7MkvxA5rdi81Cti/xlz/O8yzoXVz/+cCnk/UaWSSbzSsxlmirhnwz74uQIP6/w8zoQd0m1FtVairjG9FJ9gznxvBJn58diXmRZnC9U2Oud51L8/NuZADrht1j7YhFXsZJ4SGVc5rLsCyXlImByobgGYvNBAJOQt3Fax1VRClWVSb5SPAryPUJebKqkyBXRleJWlrO4SjKxLIusy+HPV+cfAZQvk5WIK/62SfJcLpjjpArFeZ4+8HgW3ydZnYm1jDc9f/DP7D4QiRKl3KTxXC76TD+p9Ni8KBdAQmvU7L9lmZWQZpKLTVzGaSpTUW9EVYjB4FMxU6F4n+SJWhNtAu7FpRTqJtlsMFLkQEqz79YyZ3aMRIlcnc/Xcb6SC0j3VxWvpPBPk7y+P/jH1Wkw6gmrT6HmZbKp1AEQRVkMsd2HmweQrzeQoYwz8cYIRszXcn5T1NX3xFx8G4s39Hs41O+leJMVsVhCYnj5Xvw3aNC/wQD6FCdDPBhlQiuzOC2g8llyI81KxSsxGQwyCVILJcIwnNI7KVuJoTjit/vsXizkMq7Tavxy1f/4rxMofvUwHh6upmSXvV6SbYqyEnG5gjiVtO9Q5rwuS1hUuKyrupRgQQnz2ACpW/sIRiVso5m/jtU6TWb29ZMqcvtcNPMBIu+bl3q2KYu5VO3nh+axWkOwC9hnM0C02BA3cUWUhPlwgddeD1PDRZFX0V2ZVDKaPVSwpYUUY3Fd1g2TchhlMivKh2gpY5XMkjSpHmidcijEVyIv/h2PsC8PjzSlm3i1SmWErZ2URU67LVJZAX0YdL9eXF1fnhyfRW/Pz84+XD9C0Xt7/vH9hx+vwMOWFe2dDL2R8KDrA/xEevuEJCmvbwCOngJEBfbhNVbfhXy1BxIO5zo6/9dxRBsFdiLLSGHrz9fR6WX0/qKL4PWXILBTd73LI6zHixefalXxdpfLZTJPyL8VS5ge5mRJHpMb8Xr/OD49JXDjdDDGehT+HfZxIHwFJcG7BV7v7enx1dWH9x9O3hH8PI2VSpYJCCS5quIcVuL1Li4/fLyOTs/f/gKYxkDC02J+4we9Xg92D/nf+bx3BTY+bDdvDDVsHgimeaFfv2OjhXU1D8JEFcuixB73aVxt5HzsGSY9SyItVj6+kcMwlNiDt9yNzLaGA4GHrfylN9nu5yIYff3T6Ouz0ddXu6nYGqQ7ry+Waa3WYzLdlmq8MGryrd/pU3Sxq43vIBU5DFdJxeavHDBjiROATwNXOKTWkFArHwiCvt3GoVrHR6+/48FwLe8XyUqqqpEye44IvsYnERkOsniDGLAic9/xACRJkBW5boIL1Qb7zff6XtCK6EY+9AWZB+1VgrZQYy9ogAzqCYBDClsbP5gCnKfZAQZOlsIz7s+DLTBpM7klWcaJkuI3mnxSlkXpe9pv5uxJ48Z/vlHJ7/J7AdMuH7yO2AxKI41ZnaSkGw4svtbRo2gcLdN4pR/7gkIC/cY3kDWyAOcNIIdEeqMoRav4CANt+bfhfCyW3ta87MRg28zfCX7eeTzl020GUM3WxCPikQnz3tQIWy+OIjQgJwlrLumbbIBkKPM6kyVM1we2gJjV3yB8qEzRBoAUEdu9YGqXk8rcJ5SB+MtYDJ8T/wmymTl5E058kDdQ4CNsDged9KOzCM38JkbSQCaJFYDHCVGeHE6nVgTNAIuNkG8BbESEmHtD499srZiaNMmbQrTI2o7L5fJ9ksorHfEHS6E3VxjjQyAGqfAbXeC1SbRISAib37jmM9EW4A3Yz1E8w5YHOh82gT34Auz2madpv12XMbYlnGsalXAewBrR3NaAyFTsKFxZtAS/vrPXDHmyJh4D9QrLbt7ZjzUIig0UmMu7NMnl2AOHMkdIhdmPvbpaDv7qBRQ5kUEtUsc4yXTAHakMmMJ3yby6hKOWpa8hHXYcFjDDFRCNYr0sHvEOKdHbGJmVnokk5gyGWCZxSnYk8cXmnseX79/DduaSN1ApEetn2CcJ0intwPQ4uJsXNRIdzocIJwk2ipBGVlGEoJQu+6J1nmVRVA7X9Dlsv/IrgZjHFNHIPCIRyRQW10XVfxK6+uQwGz5WsjIsmFzQoc0Kash0JQn9mwnsbxsGulCOkFuQiZk47YAui3TBXqZZojiwJPYAhtnNIil9uHA4TMWxi+JToqqouDGhzJ2F9CKmPaepHFCSkCxgYibP6cCSPXZgnb3nPZYCIW6tn1X+uQ1h/1nS5DvaiMiISFVRJe/hJLv8y1TJPZiGoebNxOBH1tLUanopwXOcAJUBj+yoz/v9ySRmVCe+zCmvYVFnG+XbmY+Y36N7UGT0/YaH7pRnrEZ7JqqOqALz6VefCgzVFzYamgST0gstBQ6AEdIMow6kJo54dJHTiZ2deEoBDVR6ei2ASeMZ6kKkjjmcmWejKcVP8m6sLPawXBr3OuZNjELNCiFfkW0h5/Hwl0qyxt7xpFnCw9IjJONtS9dEEV2+Rk+MVY87Zq2jFNPFY0g7vlm5HZ00eYzJ2Ex+0e9Eui9NPRi7jj7m0SQhWpI2q2yNkPhyGdSAX3FbgQpa41fd4n1WVxyJSNdyMRJn8S1qb+ZUUY1AkVyhurBeWxV1SS4bZTumGfy0ewalRIG4qOfJLJV9072Q5YP4hesyLAlFAlf4BnmlmQpdebHJYSGPEltni0waydp1TzwN5U07hjsNQg6BcCHdpNg4Hkf/+5wN7KHrX1z4vW6G0qx0weKHx4urGsmOGFN9VGSbFKXWwmOpNEDumjWoOzLat58xt69r5C935RTrc9IkAo7vdRMe5Auca2KtK3p2PbYjFL/ZJoQoCOsc6caNnyVQbb7qkitUqB4y/m4tdX9E0ODUkUMtcjMSt8wpwvItcQo0pqLXfsznfPbGlgtwuz8f/3YcnV9cfzj/GP12fPnh+IfTk6uds8ebkp4TEngK1NLswDyzTfDe7GhPe44IxQuGrWdjJ4J3vS+t5jBgrbHFb3VIiRjNoCZK+KlIct98CVoEWvWjjt75K1kxxt1KsEOCrdyY/ajrrj0bvyLd3MP3R50P1nXZTjcvDgH4GIy7iTfmNJ4Mn5rnVogtvj0bsyN+rh6itlXgztjzeaolovtdc9KKdobFTDl414WipcJcarJNP0DJDpx4pOmbOsrwmtrJzQDbOooiUIxpPtf5Gq02IeofLDWMEFuwh3LNrHhHTQCyj514EmNQX2jjm3wDKX0z3QWe9dpMjXoi1FfIChhykSdz45g4Z2y3WbspA53he3cznckXdeV2LrgzR9lf06YLL3hC4y7nd4uxxkvVywLzx/jhZ1mWY2fi1fW781+vqYK4HeOn9XCorLteKaLp5OmQgXNLdkwKuIuT6pVvsW0SED/sZif2m/ZqpvVnZmqEUVVE8GEVO3A99AiFjKkIZLJhCW3G9yUk8EIMD49eNZDyfi43lXi7RsS50ES5ih3tZYfI+3vJNEWXTOON2qs+MbC67XVdZdfZPeszqdgZO0Wjm2533LWpojXfzE7DIYU2quj29SKeKNDFwM0vxLrKx/xJ2+KbOjHOyNP/RT6wHPtOZyD4POqGuaLIdK6sewJF5rupNVjfZxljcajrghbj+A9cBsPT+kHCKQbZjkQnJButUvDeD4phC+SWEC3AMkbqsHCTSpQQ1Dj0naizNKcerpchZ4galhZJLvXJyhkAtTNZA7VgI9vbHFkzdPwfGWqEHaArGcKHAfLaVmgdh9uMav9HucrIbuUWZ0kN7tYkLo+mgTUv7nkZEXRTHE7eSeEOoqyIH6+gRUtt5z+BeBc8TuM/U1RpiD7sGwZTjYfGnLVj32pao7+p3Rc498ujcePeyyN4d9HWtUuPVtp+f7x2goaexlt2LbrFUaMc4NcDcfTixcvDvjgKdj8mP3T7mHYh72NIokcnFpcnw0PqAXrLMoGIoMyfilrJv57SozmTokc6lqK/MoUfqYpS0cuZrMriuozpRIDhYcYltej6zj723hNmRJBTGROQff2x7L6qlWcrS2TZSRbThjQr9m0W1W2ReJ53QWmEuI5n4SukWnEuSFD60IFyQKoF6TjQb1qJJ0ORxfM1Uqvgv8R3h0LZRloVz+qUiHKbiPVAZ0KqRvQrk99lJIf2HOjd8fXx1cn1VV+cnVz/dP4OD9fHP+iYQfkCbeotK9rkOaY1vqHIyeN9EVHuaabvem0rg2pOfNFoSLdOk8fSfdLhI+oTnjLRKKbTiQUOyWrvm3pu6trDd4fhoZE5Ei1sAj5R5YIeATmvs4iH8azSolKt2E+LnGqhgfECCEMDkvlgmZTUdcPIis/PqP7Tx+BV4eAT9xqhuCvKG1mqRuRcKJE9grEpQrHvzHlhmNBJkB4D5GTK0mV5lnS868wxq6XvxAU3ygoKsHaJyNHHaZzNFrGYj4Q/eGp588kRVYWT4TTQf+i3Hno9FUB8GLgNO14P2MqQrWtuqC3OywoCQ49ewihCBUf1SBS1wZE/TTSSqfh2vGcnEOOaPj0Mp+1kvWQzWxwcaHFNw3iDLE6vODAHNzdSbeKcxYesR7M3aSQ88pX4VgwD+w7XShJUz0jYNouMtTQUjHFlMcSBtPl2TN7XyIsPlUxDhp/D43JVk9Vc8Bd/IfVBPFKWcRQtinkUUQVIx3V0Usnd4nEz+TK+e9dO+Emmm/cWNHDohfECIdQQopMgW+zAGVUPGzm+4J5YKf9dJyW8dptafWY+5SHel00o/ywt07XqTrfHV/TmexYkeBYReYPBnJrsn8cVrguqgjj/XM1K6pFxpeY9i5mKYxjnukjgGMZmt5kqNGgJ0Nn4H3JIFyOALKeCa+x9+/wEc0/iP4ZHIprWC7lvXsvmZPosDr6L0Z3JwkQi/LwJoPJ0YFt6w2enUeVsZjUnsu3c9pDWthLHr1Z/YAebNM4ddSEVHg7gL8riFsnAtP+k82z/rbG9xi4wqmL41+FLuudT2JgFT3B8v5ohCvMZn3hblHKAfMP/aXA49KmNwQ/LIHheVexWOvLS9IeDWUzJPH/n7Oye4gwC/vP44L40zj+jhDRRFabFc/ZLyH2xqKjCzvUsWxd0C0C3PZkxErLuZiJZ5/ocZTNxmSNuGk7JgPSBONHUqqQx9ppNbcOdB7PatjprSpWheDN2Yexb666dgl9TkvZAXM/IaoTvmcT/6k7KHBgJd0diQc+0W580+sG/e2uBSbdHIfxK9xf0em+RLMazlMr2yYLjC+c99jy22f5Tm58tOmfPV79eeMGUOsd4esEAaq3vgPl/H7w03YLmdpURRPMOcn7LgpUsK4qLhKdsaIR1foPKK+dDsqZHrrhyx/vnZgG/mfgZ+S+9d5Yz05Xc2sadFdt0NxJb404NsmBnjMf0/zme8+0Flqd+cmRqwbRM+fMEWZahaAixU4yMU5x2DM8MtmswHQgjDReGJLJtKHS4sVzs3La3wfSo/u+K6MxchDP86lW1QjEoGqH8B4JpPrvisKLoiEBfurObi18opzSCRVK0KLDFOBpMzf7grBd02xTUyeSthXTP0XWv8ekZmMvrk0Pg1nKp2nX9t/YL1K+nL8f//PGH0uuUFEaSugp8eraJ5VVJXkvaZcZRj/Y69M7Mr8R5LgdUdJU19pZvXfyoFbd1gbE+tOMrMDSBj/KpXDIMho+OnUtzyJcLX+ugc4mGN+6EHOJ0z5Ev68Pmwr7dVc4R5OeEb67Xdn2wU5Z8DycJzniYYsPoUULe7+Tbe8qsRwj7bXPa6ZsWVZxyczbzn61RdK3D+4OwU/b28rvDQ6fC4J4FFSUaYCcMG//3v/+zZTKjcLjcibcXvw7WdNdXF7Bqw+cW20fM7kRT0G0btne6bHD2Id8vovDch+ZoAoQKgQTdq0a/Jxv/icyQJAwfqdOW2Gbjb2nh7boZw67bg9WrNhoRW2aF/CmJgceC0ctFIwkIOGwYgH+h31qMo9fhEWSz7reRZcvcuE7H2kjXGJwqqdWGdRKmdHNC90AMp48ivg5MfOW5NWiSgd+tYp8vYgPaoaku3wWX6+2pgT4fOKCi/mC+qZN8WeijAnpql9I9/wD7OfXLfDquspf5Rh6pDfTsbT3WD0GQiggd3xjjGU5U1whJoh5Vy1ws6gtd+oDp6bmuPdXWB6r6Dsae01pSFyY3V4v2JicEFHFR1Onv7dkrrq03l9R//u1M9fWlugw/W3uyb22D7yCp9roki9xcuA6v+Y7QRVGkJ/dyXiOp9FGZR6Y/Mm5dAqljA7BWHXxVHIGGRkNVzzIowN7N0G2CL7ic0emV8GqnnV0MYvTFsh2rqOmv+szfY4vXDUd7P39MCMzli0fnJDBi6oUbSH2lw8xiyzfdUNts7t5d0rKl+6JTvvtIb3xEztdNDwPqZzg6XdCZhthquF23R3qoowplc/Q1oCx663SR+y2PO83ZsNfrJXShjAw3ijjaRhH1PKLIs7d16e7a1YOqZHaCKsDnjgiCyv8DtIAc7Q=="
}
EXPECTED_SHA256 = {
 "kaggle_environment_smoke.py": "ae1c29176e8cbd0de0324841d29fae757240ee958181de209c92fc79e05fe2de",
 "e1_memory_feasibility.py": "e09eb6b5f18fb89142b9b258c7fdeab4d74818b68ef4856879cd58215740e69a",
 "sgbr_c01_e1_integrated.py": "55c60b728d8e7141217e39474675b4c4a317aa7d79b1bba376b5a8b9d254552b",
 "summarize_e1.py": "1fb8686f26924aa2c8c04c83c1e6c671483daa3a366356d2486e9c0bb9ca054c",
 "run_matrix.py": "42e394f6121d23419878c30fb20f5561ea7983cb9fdbe652292c0be229469c0f"
}
for name, blob in EMBEDDED.items():
    data = zlib.decompress(base64.b64decode(blob))
    if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256[name]:
        raise RuntimeError(f"Embedded script {name} is corrupted")
    (SCRIPTS / name).write_bytes(data)
    print(f"{name:32s} sha256 {EXPECTED_SHA256[name]}")
sys.dont_write_bytecode = True
sys.path.insert(0, str(SCRIPTS))


In [ ]:
import datetime, json, shutil, zipfile

assert 1 <= SHARD <= NUM_SHARDS, "Set SHARD between 1 and NUM_SHARDS"
STARTED = datetime.datetime.now(datetime.timezone.utc)
RESULTS = WORK / "results"
ARTIFACTS = WORK / "artifacts"
CACHE = WORK / ".sgbr-cache"
for directory in (RESULTS, ARTIFACTS, CACHE):
    directory.mkdir(parents=True, exist_ok=True)
if RESUME_ZIP:
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        archive.extractall(RESULTS)
    print("Restored earlier results from", RESUME_ZIP)

import sgbr_c01_e1_integrated as pipeline

pipeline.stage("Provisioning JDK 11 and Maven (checksum-verified)")
JAVA, MVN, TOOLS = pipeline.provision(CACHE)
pipeline.stage("Fetching the authors' pinned MOA source (f62f7b4)")
UPSTREAM, PIN = pipeline.upstream_checkout(CACHE)
pipeline.stage("Building the MOA jar (about 5-10 minutes)")
JAR, BUILD = pipeline.build_or_reuse(UPSTREAM, ARTIFACTS, MVN, TOOLS, PIN)
print(f"Jar SHA-256: {BUILD['sha256']}")
(RESULTS / "manifest").mkdir(parents=True, exist_ok=True)
(RESULTS / "manifest" / f"E1_shard{SHARD}_toolchain.json").write_text(json.dumps(
    {"shard": SHARD, "num_shards": NUM_SHARDS, "jobs": JOBS, "xmx": XMX, "started_at": STARTED.isoformat(),
     "build": BUILD, "tools": TOOLS, "script_sha256": EXPECTED_SHA256}, indent=1, default=str))


In [ ]:
import os, signal, subprocess, sys, threading

elapsed_h = (datetime.datetime.now(datetime.timezone.utc) - STARTED).total_seconds() / 3600
budget_s = int((SESSION_BUDGET_HOURS - elapsed_h) * 3600)
arguments = [
    sys.executable, str(SCRIPTS / "run_matrix.py"), "--upstream", str(UPSTREAM), "--java", str(JAVA),
    "--jar", str(JAR), "--results", str(RESULTS), "--data-cache", str(WORK / "data"),
    "--exp", "E1", "--plan", "e1-approved", "--num-shards", str(NUM_SHARDS), "--shard", str(SHARD),
    "--jobs", str(JOBS), "--xmx", XMX,
]
pipeline.stage(f"Running E1 shard {SHARD}/{NUM_SHARDS}; budget {budget_s / 3600:.1f} h")
process = subprocess.Popen(arguments, cwd=SCRIPTS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)


def stop_at_budget():
    print("Session budget reached: stopping. Unfinished runs restart on resume.", flush=True)
    os.killpg(process.pid, signal.SIGTERM)


timer = threading.Timer(max(budget_s, 60), stop_at_budget)
timer.start()
try:
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    timer.cancel()
    process.wait()
print("run_matrix exit code:", process.returncode)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(SCRIPTS / "summarize_e1.py"), "--upstream", str(UPSTREAM),
                "--results", str(RESULTS)], cwd=SCRIPTS, check=False)
archive = shutil.make_archive(str(WORK / f"E1_shard{SHARD}"), "zip", RESULTS)
print(f"Download {archive} from the Output tab and send it back.")
